# Project 01 — What are customers really saying?

An online clothing shop has a thousand reviews and nobody with time to read them.
The support team wants two things: to know **what people keep talking about**, and,
when a new review arrives, to see **the reviews most like it** — so a reply can be
written by someone who has seen that kind of feedback before.

You will build both, with **a model running on your own machine** and a **vector
database**. No API key, no account, nothing sent anywhere.

## The data

`data/reviews.csv` — 1,000 real reviews, anonymised by their publisher and released
into the public domain (CC0). You need one column:

| Column | What it holds |
|---|---|
| `Review Text` | What the customer wrote about the product and the purchase |

## What you deliver

| Task | Store it in | What it is |
|---|---|---|
| 1 | `embeddings` | one vector per review |
| 2 | `embeddings_2d` | the same reviews as 2-D points, and a plot of them |
| 3 | `topic_reviews` | a dict: topic → the reviews closest to it |
| 4 | `most_similar_reviews` | the 3 reviews most like *"Absolutely wonderful - silky and sexy and comfortable"* |

Each task ends with a check cell. The checks are **not counted** toward your marks.

## Before you start — two installs

```bash
uv sync --extra projects                 # ChromaDB, scikit-learn, pandas, matplotlib
ollama pull nomic-embed-text             # a 274 MB embedding model
```

**Only 8 GB of RAM?** Run it on Colab instead: `demos/04_ollama_on_colab.ipynb`
sets up Ollama there — then `ollama pull nomic-embed-text` in the same way.

In [ ]:
# manual-run: needs a local Ollama server, an embedding model, and the `projects` extra
import json
import sys
import urllib.error
import urllib.request
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

try:
    import chromadb
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from scipy.spatial import distance
    from sklearn.manifold import TSNE
except ImportError as error:
    raise SystemExit(f"missing {error.name!r}. Run: uv sync --extra projects") from None

MODEL = "nomic-embed-text"
OLLAMA = "http://localhost:11434"

try:
    with urllib.request.urlopen(f"{OLLAMA}/api/tags", timeout=5) as response:
        pulled = {m["name"].split(":")[0] for m in json.loads(response.read())["models"]}
except (urllib.error.URLError, OSError):
    raise SystemExit("no Ollama server on localhost:11434. Start it with: ollama serve") from None
if MODEL not in pulled:
    raise SystemExit(f"the model is not pulled yet. Run: ollama pull {MODEL}")

from bootcamp_agent.bonus import bonus
from bootcamp_agent.projects import clothing_reviews  # noqa: F401  (registers the checks)

print(f"ready: chromadb {chromadb.__version__}, model {MODEL}")

## Look at the data before you trust it

Real data is never quite what the column name promises.

In [ ]:
reviews = pd.read_csv(ROOT / "projects/01-clothing-reviews/data/reviews.csv")
print(f"{len(reviews)} rows")
print(f"{reviews['Review Text'].isna().sum()} with no review text at all")
reviews[["Review Text", "Rating", "Class Name"]].head()

Some rows have **no text**. An empty review still gets an embedding if you ask for
one — a vector that means nothing, sitting among the real ones and turning up in
searches. Decide what to do with them before you embed anything.

---

## Task 1 — Create the embeddings

**Store in `embeddings`** · about 10 minutes · check: `project-01-e1`

An embedding turns text into a list of numbers, placed so that texts which *mean*
similar things sit close together. That closeness is what every later task uses.

**Done when** `embeddings` holds one vector per review that has text, and the check
is green.

The helper below talks to your local model. It is the equivalent of
`client.embeddings.create()` — you call it with a list of texts and get back a list
of vectors.

<details><summary><b>Hint</b></summary>

- Take the `Review Text` column and drop the empty ones first: `.dropna()`
- Sending 958 texts at once is slow and can time out. Send them in batches of 64.
- `embed(batch)` returns a list; extend `embeddings` with it.

</details>

In [ ]:
def embed(texts: list[str]) -> list[list[float]]:
    """Embeddings for a batch of texts, from the model on your machine."""
    request = urllib.request.Request(
        f"{OLLAMA}/api/embed",
        data=json.dumps({"model": MODEL, "input": texts}).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(request, timeout=300) as response:
        return json.loads(response.read())["embeddings"]


vector = embed(["Runs small, order a size up."])[0]
print(f"one review -> {len(vector)} numbers, starting {[round(x, 3) for x in vector[:4]]}")

In [ ]:
# Task 1 — start coding here

review_texts = []   # TODO(you): the review texts, without the empty ones
embeddings = []     # TODO(you): one embedding per text, in batches

bonus("project-01-e1", embeddings)

---

## Task 2 — Reduce to 2-D and look at it

**Store in `embeddings_2d`** · about 10 minutes · check: `project-01-e2`

Each embedding has hundreds of numbers, and nobody can look at 768 dimensions.
t-SNE squeezes them down to two while trying to keep neighbours as neighbours, so
you can *see* whether similar reviews really do sit together.

**Done when** `embeddings_2d` is one `(x, y)` point per review, you have a scatter
plot, and the check is green.

<details><summary><b>Hint</b></summary>

- `TSNE(n_components=2, random_state=0).fit_transform(np.array(embeddings))`
- `plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], s=6)`
- Colour the points by `Rating` and see whether the unhappy reviews cluster.

</details>

**Read the plot honestly.** t-SNE keeps *local* neighbourhoods. The distance between
two far-apart clusters means very little, so do not read one as "twice as different".

In [ ]:
# Task 2 — start coding here

embeddings_2d = None   # TODO(you): t-SNE down to 2 dimensions

# TODO(you): a scatter plot of embeddings_2d

bonus("project-01-e2", embeddings_2d)

---

## Task 3 — What do people keep talking about?

**Store in `topic_reviews`** · about 15 minutes · check: `project-01-e3`

Embed a few **topic words** — `quality`, `fit`, `style`, `comfort` — the same way
you embedded the reviews. Then, for each topic, find the reviews whose embeddings
are closest to it.

**Done when** `topic_reviews` maps at least three topics to real reviews, and the
check is green.

<details><summary><b>Hint</b></summary>

- `embed(["quality", "fit", "style", "comfort"])` gives one vector per topic.
- `distance.cosine(a, b)` is **0** for identical direction and grows as they differ.
- Sort the reviews by their distance to a topic and keep the closest three.

</details>

**Now read what came back.** Does every review under `quality` actually talk about
quality? Write down how many of the three do. You will need that number.

In [ ]:
# Task 3 — start coding here

topics = ["quality", "fit", "style", "comfort"]
topic_reviews = {}   # TODO(you): topic -> the 3 closest reviews

bonus("project-01-e3", topic_reviews)

### Stand above it — read the whole review before you judge

Look at what came back under `quality`. If you only read the first line of each
review, some of them seem to be about colour, or styling — and it is tempting to
decide the search is bad.

**Read them in full before you decide.** The cell below prints where the word
actually appears in each one.

In [ ]:
if not topic_reviews.get("quality"):
    raise SystemExit("Finish Task 3 first: this reads what topic_reviews found for 'quality'.")

for review in topic_reviews["quality"]:
    at = review.lower().find("quality")
    opening = review[:60].replace("\n", " ")
    if at < 0:
        print(f"- {opening}...\n    (never says 'quality')\n")
    else:
        print(f"- {opening}...")
        print(f"    ...{review[max(0, at - 50):at + 45]}...\n")

Some of those reviews do not *open* with quality — they are **complaints** about it,
further in. The search was right. A glance at the first line would have said it
was wrong.

### And measure a change instead of assuming it

This model's documentation says to put `search_document: ` before the texts you
store and `search_query: ` before what you search with. Documentation says to do
it, so it must help — right? Measure it.

In [ ]:
if len(embeddings) != len(review_texts) or not review_texts:
    raise SystemExit("Finish Task 1 first: this compares against your embeddings.")


def top3(query_vector, document_vectors):
    order = sorted(range(len(review_texts)), key=lambda i: distance.cosine(document_vectors[i], query_vector))
    return [review_texts[i] for i in order[:3]]


prefixed = []
for start in range(0, len(review_texts), 64):
    prefixed.extend(embed(["search_document: " + t for t in review_texts[start:start + 64]]))

for label, hits in (
    ("without prefixes", top3(embed(["quality"])[0], embeddings)),
    ("with prefixes", top3(embed(["search_query: quality"])[0], prefixed)),
):
    mentions = sum("quality" in text.lower() for text in hits)
    print(f"{label}: {mentions} of 3 talk about quality")
    for text in hits:
        at = text.lower().find("quality")
        print("   -", text[max(0, at - 30):at + 50].replace("\n", " "))
    print()

On this data, for this topic, **both find reviews about quality.** What changed is
*which* ones: without prefixes you get complaints and praise mixed; with them,
mostly praise.

That is not "better" or "worse" until you say what the support team needs. If the
point is to find unhappy customers, the version that surfaced complaints was more
useful — and the documented setting would have hidden them.

Two habits worth more than any setting: **read the whole result**, and **measure
before you claim an improvement**.

---

## Task 4 — Find the reviews most like this one

**Store in `most_similar_reviews`** · about 15 minutes · check: `project-01-e4`

Searching 958 vectors one by one is fine. Searching a million is not, and that is
what a **vector database** is for: it stores the embeddings once and answers
*"what is nearest to this?"* quickly. You will use ChromaDB.

Write a function that returns the **3 reviews most similar** to a given review, and
apply it to:

> *Absolutely wonderful - silky and sexy and comfortable*

**Done when** `most_similar_reviews` is a list of three review texts and the check
is green.

The class below plugs your local model into ChromaDB, so ChromaDB can embed text
itself. It uses the prefixes the model's documentation recommends — you have just
seen that they change *which* neighbours come back, so keep that in mind when you
read the results.

<details><summary><b>Hint</b></summary>

- `client = chromadb.Client()` — an in-memory database, gone when the notebook stops
- `client.create_collection(name=..., embedding_function=LocalEmbeddings())`
- `collection.add(documents=..., ids=...)` — ids are strings, one per review
- `collection.query(query_texts=[...], n_results=...)`

</details>

**Look closely at the first thing it returns.** Then decide whether that is what the
support team asked for.

In [ ]:
from chromadb import Documents, EmbeddingFunction, Embeddings


class LocalEmbeddings(EmbeddingFunction):
    """ChromaDB's seam for 'turn text into vectors', pointed at the local model."""

    def __init__(self) -> None:
        pass

    def __call__(self, input: Documents) -> Embeddings:
        return embed(["search_document: " + text for text in input])

    def embed_query(self, input: Documents) -> Embeddings:
        return embed(["search_query: " + text for text in input])

    @staticmethod
    def name() -> str:
        return "local-nomic-embed-text"

    def get_config(self) -> dict:
        return {}

    @staticmethod
    def build_from_config(config: dict) -> "LocalEmbeddings":
        return LocalEmbeddings()


print("embedding function ready")

In [ ]:
# Task 4 — start coding here

client = chromadb.Client()
# TODO(you): create a collection with LocalEmbeddings(), and add every review


def find_similar_reviews(text: str, n: int = 3) -> list[str]:
    """The n reviews most similar to `text`."""
    return []   # TODO(you)


most_similar_reviews = find_similar_reviews("Absolutely wonderful - silky and sexy and comfortable")
bonus("project-01-e4", most_similar_reviews)

---

## What you just built

- **Embeddings** turn meaning into position. Close means similar.
- **Dimensionality reduction** makes that position something you can look at — for
  neighbourhoods, not for distances between far-apart groups.
- **Topics** are just more embeddings. Search for them the way you search for anything.
- **Read the whole result, and measure a change** — a first-line glance and a
  documented setting both told a story the data did not support.
- **A vector database** stores the vectors once and answers "what is nearest" fast —
  and the nearest thing to a review is always **that review**.

## Stuck? Ask the course

The coach answers from the course pages, offline, with no key.

In [ ]:
from bootcamp_agent.coach import coach

coach("embeddings cosine distance vector database nearest neighbour", top_k=1, max_chars=700)

## Your turn

Not checked. Pick one.

1. **Replace the word with a sentence.** Search `search_query: reviews about fabric
   quality and how well it is made` instead of `quality`. Better or worse? Count.
2. **Find the unhappy customers.** Filter the plot to `Rating <= 2`. Do they cluster?
   What do their nearest neighbours say?
3. **Answer a brand-new review.** Write one that is not in the data, pass it to
   `find_similar_reviews`, and read what a support agent would see.